<a href="https://colab.research.google.com/github/hOshi123456/Proyecto_Inteligencia_Artificial_MarianRomero/blob/main/Proyecto_IA_Ventas_Mistral_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#Instalar dependencias
!pip install -q pandas numpy python-dotenv requests

In [8]:
import pandas as pd
import numpy as np
import os
import requests

print("Entorno preparado correctamente ✅")

Entorno preparado correctamente ✅


In [9]:
#Adjuntar archivo del dataset descargado
from google.colab import files

uploaded = files.upload()

Saving sales_data_sample.csv to sales_data_sample.csv


In [10]:
#Ver el nomnbre del archivo que se subió
import os

os.listdir()

['.config', 'sales_data_sample.csv', 'sample_data']

In [11]:
#Cargar el dataset de ventas#
archivo = "sales_data_sample.csv"

df = pd.read_csv(archivo, encoding="latin1")

print("Dataset cargado correctamente ✅")
print("Filas y columnas:", df.shape)

Dataset cargado correctamente ✅
Filas y columnas: (2823, 25)


In [12]:
# Ver las primeras 5 filas del dataset

df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,8/25/2003 0:00,Shipped,3,8,2003,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,10/10/2003 0:00,Shipped,4,10,2003,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [13]:
# PASO 4: Explorar el dataset

print("Columnas del dataset:")
print(df.columns.tolist())

print("\nInformación general:")
df.info()

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nCantidad de filas duplicadas:")
print(df.duplicated().sum())

Columnas del dataset:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ORDERNUMBER       2823 non-null   int64  
 1   QUANTITYORDERED   2823 non-null   int64  
 2   PRICEEACH         2823 non-null   float64
 3   ORDERLINENUMBER   2823 non-null   int64  
 4   SALES             2823 non-null   float64
 5   ORDERDATE         2823 non-null   object 
 6   STATUS            2823 non-null   object 
 7   QTR_ID            2823 non-null   int64  
 8   MONTH_ID          2823 non-null   int64  

In [14]:
#Normalización y limpieza básica del dataset

# Creamos una copia para no dañar el dataset original
df_limpio = df.copy()

# Normalizar nombres de columnas: minúsculas y sin espacios
df_limpio.columns = (
    df_limpio.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
)

# Quitar espacios extra en columnas de texto
columnas_texto = df_limpio.select_dtypes(include=["object"]).columns

for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].astype(str).str.strip()

# Convertir orderdate a formato fecha
df_limpio["orderdate"] = pd.to_datetime(df_limpio["orderdate"], errors="coerce")

# Rellenar valores nulos en columnas específicas
df_limpio["addressline2"] = df_limpio["addressline2"].replace("nan", "No especificado")
df_limpio["state"] = df_limpio["state"].replace("nan", "No especificado")
df_limpio["postalcode"] = df_limpio["postalcode"].replace("nan", "No especificado")
df_limpio["territory"] = df_limpio["territory"].replace("nan", "No especificado")

# Verificar resultado
print("Dataset normalizado correctamente ✅")
print("Filas y columnas:", df_limpio.shape)

print("\nValores nulos restantes:")
print(df_limpio.isnull().sum())

df_limpio.head()

Dataset normalizado correctamente ✅
Filas y columnas: (2823, 25)

Valores nulos restantes:
ordernumber         0
quantityordered     0
priceeach           0
orderlinenumber     0
sales               0
orderdate           0
status              0
qtr_id              0
month_id            0
year_id             0
productline         0
msrp                0
productcode         0
customername        0
phone               0
addressline1        0
addressline2        0
city                0
state               0
postalcode          0
country             0
territory           0
contactlastname     0
contactfirstname    0
dealsize            0
dtype: int64


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,No especificado,NYC,NY,10022,USA,No especificado,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,No especificado,Reims,No especificado,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,No especificado,Paris,No especificado,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,...,78934 Hillside Dr.,No especificado,Pasadena,CA,90003,USA,No especificado,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,...,7734 Strong St.,No especificado,San Francisco,CA,No especificado,USA,No especificado,Brown,Julie,Medium


In [15]:
#Exportar el dataset limpio

nombre_archivo_limpio = "ventas_limpio.csv"

df_limpio.to_csv(nombre_archivo_limpio, index=False, encoding="utf-8")

print("Archivo exportado correctamente ✅")
print("Nombre del archivo:", nombre_archivo_limpio)

Archivo exportado correctamente ✅
Nombre del archivo: ventas_limpio.csv


In [16]:
from google.colab import files

files.download("ventas_limpio.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
#Probar conexión con Mistral usando API Key protegida

from google.colab import userdata
import requests

MistralApi = userdata.get("MistralApi")

url = "https://api.mistral.ai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {MistralApi}",
    "Content-Type": "application/json"
}

data = {
    "model": "mistral-small-latest",
    "messages": [
        {
            "role": "user",
            "content": "Hola, responde en español: ¿qué es un dataset de ventas?"
        }
    ],
    "temperature": 0.3
}

response = requests.post(url, headers=headers, json=data)

print("Código de estado:", response.status_code)

if response.status_code == 200:
    respuesta = response.json()
    print("\nRespuesta de Mistral:")
    print(respuesta["choices"][0]["message"]["content"])
else:
    print("\nError:")
    print(response.text)

Código de estado: 200

Respuesta de Mistral:
¡Hola! Un **dataset de ventas** es un conjunto de datos estructurados que contiene información relacionada con las transacciones comerciales de una empresa o negocio. Estos datos suelen incluir detalles como:

- **Productos o servicios vendidos** (nombres, categorías, precios).
- **Clientes** (identificadores, datos demográficos, historial de compras).
- **Fechas y horarios** de las transacciones.
- **Cantidades vendidas** y montos totales.
- **Métodos de pago** (efectivo, tarjeta, transferencia, etc.).
- **Ubicaciones** (tiendas físicas, online, regiones geográficas).
- **Descuentos o promociones** aplicadas.

### Ejemplo de estructura:
| ID_Venta | Fecha       | Producto   | Cliente_ID | Cantidad | Precio_Unitario | Total |
|----------|-------------|------------|------------|----------|-----------------|-------|
| 1001     | 2023-10-01  | Laptop     | C001       | 1        | 1200            | 1200  |
| 1002     | 2023-10-02  | Mouse      |

In [18]:
#Crear resumen del dataset limpio para Mistral

resumen_dataset = f"""
Resumen del dataset de ventas:

Cantidad de filas: {df_limpio.shape[0]}
Cantidad de columnas: {df_limpio.shape[1]}

Columnas disponibles:
{', '.join(df_limpio.columns)}

Total de ventas: {df_limpio['sales'].sum():,.2f}

Cantidad total de productos vendidos:
{df_limpio['quantityordered'].sum()}

Cantidad de órdenes únicas:
{df_limpio['ordernumber'].nunique()}

Países presentes en el dataset:
{', '.join(df_limpio['country'].dropna().unique())}

Líneas de productos:
{', '.join(df_limpio['productline'].dropna().unique())}

Estados de las órdenes:
{', '.join(df_limpio['status'].dropna().unique())}

Ventas por línea de producto:
{df_limpio.groupby('productline')['sales'].sum().sort_values(ascending=False).to_string()}

Ventas por país:
{df_limpio.groupby('country')['sales'].sum().sort_values(ascending=False).head(10).to_string()}

Ventas por tamaño de trato:
{df_limpio.groupby('dealsize')['sales'].sum().sort_values(ascending=False).to_string()}
"""

print(resumen_dataset)


Resumen del dataset de ventas:

Cantidad de filas: 2823
Cantidad de columnas: 25

Columnas disponibles:
ordernumber, quantityordered, priceeach, orderlinenumber, sales, orderdate, status, qtr_id, month_id, year_id, productline, msrp, productcode, customername, phone, addressline1, addressline2, city, state, postalcode, country, territory, contactlastname, contactfirstname, dealsize

Total de ventas: 10,032,628.85

Cantidad total de productos vendidos:
99067

Cantidad de órdenes únicas:
307

Países presentes en el dataset:
USA, France, Norway, Australia, Finland, Austria, UK, Spain, Sweden, Singapore, Canada, Japan, Italy, Denmark, Belgium, Philippines, Germany, Switzerland, Ireland

Líneas de productos:
Motorcycles, Classic Cars, Trucks and Buses, Vintage Cars, Planes, Ships, Trains

Estados de las órdenes:
Shipped, Disputed, In Process, Cancelled, On Hold, Resolved

Ventas por línea de producto:
productline
Classic Cars        3919615.66
Vintage Cars        1903150.84
Motorcycles    

In [19]:
#Función para preguntar a Mistral sobre el dataset

from google.colab import userdata
import requests

MistralApi = userdata.get("MistralApi")

url = "https://api.mistral.ai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {MistralApi}",
    "Content-Type": "application/json"
}

def preguntar_mistral(pregunta):
    prompt = f"""
Eres un asistente de inteligencia artificial especializado en análisis de datos.

Debes responder únicamente usando la información del siguiente resumen del dataset de ventas.

{resumen_dataset}

Pregunta del usuario:
{pregunta}

Responde en español, de forma clara y breve.
"""

    data = {
        "model": "mistral-small-latest",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.3
    }

    response = requests.post(url, headers=headers, json=data)

    if response.status_code == 200:
        respuesta = response.json()
        return respuesta["choices"][0]["message"]["content"]
    else:
        return f"Error {response.status_code}: {response.text}"

In [20]:
pregunta = "¿Cuál es la línea de producto con más ventas?"
respuesta = preguntar_mistral(pregunta)

print(respuesta)

La línea de producto con más ventas es **Classic Cars**, con un total de **3,919,615.66**.


In [21]:
# Limpieza completa de paquetes conflictivos de LangChain
!pip uninstall -y langchain langchain-core langchain-community langchain-experimental langchain-mistralai langchain-text-splitters langchain-classic langgraph langgraph-prebuilt langgraph-checkpoint

Found existing installation: langchain 1.2.15
Uninstalling langchain-1.2.15:
  Successfully uninstalled langchain-1.2.15
Found existing installation: langchain-core 1.3.1
Uninstalling langchain-core-1.3.1:
  Successfully uninstalled langchain-core-1.3.1
Found existing installation: langgraph 1.1.9
Uninstalling langgraph-1.1.9:
  Successfully uninstalled langgraph-1.1.9
Found existing installation: langgraph-prebuilt 1.0.10
Uninstalling langgraph-prebuilt-1.0.10:
  Successfully uninstalled langgraph-prebuilt-1.0.10
Found existing installation: langgraph-checkpoint 4.0.2
Uninstalling langgraph-checkpoint-4.0.2:
  Successfully uninstalled langgraph-checkpoint-4.0.2


In [22]:
# Instalación estable para create_pandas_dataframe_agent + Mistral
!pip install -U \
  "langchain<1.0" \
  "langchain-core<1.0" \
  "langchain-community<1.0" \
  "langchain-experimental<1.0" \
  "langchain-mistralai<1.0" \
  "langchain-text-splitters<1.0" \
  tabulate \
  "requests==2.32.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-experimental to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: tabulate
    Found existing installation: tabulate 0.9.0
    Uninstalling tabulate-0.9.0:
      Successfully 

In [23]:
#Este import es el que corresponde para el agente de Pandas según la documentación actual de LangChain
import os
import getpass
import pandas as pd

from langchain_mistralai import ChatMistralAI
from langchain_experimental.agents import create_pandas_dataframe_agent

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [24]:
#Para verificar que no haya conflicto (ya me creó varios)
!pip check

ipython 7.34.0 requires jedi, which is not installed.


In [25]:
# Verificar qué DataFrames existen actualmente en memoria
dataframes = {
    nombre: valor
    for nombre, valor in globals().items()
    if isinstance(valor, pd.DataFrame)
}

if len(dataframes) == 0:
    print("⚠️ No hay ningún DataFrame cargado actualmente.")
    print("Tenés que volver a ejecutar las celdas donde cargaste y limpiaste el dataset.")
else:
    print("✅ DataFrames encontrados:\n")
    for nombre, dataframe in dataframes.items():
        print(f"- {nombre}: {dataframe.shape[0]} filas x {dataframe.shape[1]} columnas")

✅ DataFrames encontrados:

- _: 5 filas x 25 columnas
- __: 5 filas x 25 columnas
- df: 2823 filas x 25 columnas
- _12: 5 filas x 25 columnas
- df_limpio: 2823 filas x 25 columnas
- _14: 5 filas x 25 columnas


In [26]:
# Usamos el dataset limpio como DataFrame principal para el agente
df = df_limpio.copy()

print("✅ Dataset limpio asignado a df")
print("Tamaño del dataset:", df.shape)

df.head()

✅ Dataset limpio asignado a df
Tamaño del dataset: (2823, 25)


,ordernumber,quantityordered,priceeach,orderlinenumber,sales,orderdate,status,qtr_id,month_id,year_id,...,addressline1,addressline2,city,state,postalcode,country,territory,contactlastname,contactfirstname,dealsize
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,No especificado,NYC,NY,10022,USA,No especificado,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,No especificado,Reims,No especificado,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,No especificado,Paris,No especificado,75508,France,EMEA,Da Cunha,Daniel,Medium
3,10145,45,83.26,6,3746.70,2003-08-25,Shipped,3,8,2003,...,78934 Hillside Dr.,No especificado,Pasadena,CA,90003,USA,No especificado,Young,Julie,Medium
4,10159,49,100.00,14,5205.27,2003-10-10,Shipped,4,10,2003,...,7734 Strong St.,No especificado,San Francisco,CA,No especificado,USA,No especificado,Brown,Julie,Medium


In [27]:
# Ver columnas disponibles en el dataset
print("Columnas disponibles:\n")

for columna in df.columns:
    print("-", columna)

Columnas disponibles:

- ordernumber
- quantityordered
- priceeach
- orderlinenumber
- sales
- orderdate
- status
- qtr_id
- month_id
- year_id
- productline
- msrp
- productcode
- customername
- phone
- addressline1
- addressline2
- city
- state
- postalcode
- country
- territory
- contactlastname
- contactfirstname
- dealsize


In [28]:
import os
import getpass

# Si tu API está guardada con el nombre MistralApi, la cargamos ahí
if "MistralApi" not in os.environ:
    os.environ["MistralApi"] = getpass.getpass("Ingresá tu API key de Mistral: ").strip()

# Copiamos la clave al nombre que LangChain espera
os.environ["MISTRAL_API_KEY"] = os.environ["MistralApi"]

print("✅ API key configurada correctamente para LangChain")

✅ API key configurada correctamente para LangChain


In [29]:
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0
)

print("✅ Modelo Mistral configurado correctamente")

✅ Modelo Mistral configurado correctamente


In [30]:
#Crear el modelo mistral
llm = ChatMistralAI(
    model="mistral-large-latest",
    temperature=0
)

print("✅ Modelo Mistral configurado correctamente")

✅ Modelo Mistral configurado correctamente


In [31]:
prefix = """
Eres un agente de análisis de datos especializado en ventas.

Estás trabajando con un DataFrame de Pandas llamado df.
El dataset contiene información de pedidos, ventas, productos, países, clientes y estados de pedidos.

Columnas importantes:
- ordernumber: número de pedido
- quantityordered: cantidad ordenada
- priceeach: precio unitario
- sales: monto total de venta
- orderdate: fecha del pedido
- status: estado del pedido
- productline: línea de producto
- customername: nombre del cliente
- country: país
- territory: territorio
- dealsize: tamaño de la venta

Reglas:
- Responde siempre en español.
- Usa únicamente los datos reales del DataFrame.
- No inventes columnas ni valores.
- Si necesitás hacer cálculos, usa Python y Pandas.
- Si una pregunta no se puede responder con el dataset, dilo claramente.
- Cuando entregues un resultado, explica brevemente qué analizaste.
- Si no podés responder, decilo claramente.
"""

In [32]:
agent = create_pandas_dataframe_agent(
    llm,
    df,
    prefix=prefix,
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
    agent_type="tool-calling"
)

print("✅ Agente creado correctamente")

✅ Agente creado correctamente


/usr/local/lib/python3.12/dist-packages/langchain_experimental/agents/agent_toolkits/pandas/base.py:283: UserWarning: Received additional kwargs {'handle_parsing_errors': True} which are no longer supported.
  warnings.warn(


In [33]:
try:
    respuesta = agent.invoke({
        "input": "Dame un resumen general del dataset de ventas."
    })

    print(respuesta["output"])

except Exception as e:
    print("❌ Tipo de error:", type(e).__name__)
    print("❌ Error completo:")
    print(e)

    if hasattr(e, "response") and e.response is not None:
        print("\n📌 Código HTTP:", e.response.status_code)
        print("\n📌 Detalle de Mistral:")
        print(e.response.text)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Obtener información general del dataset: cantidad de filas, columnas, tipos de datos y valores no nulos.\ndf_info = df.info()\n\n# Resumen estadístico de las columnas numéricas.\nstats_num = df.describe(include=['int64', 'float64'])\n\n# Resumen estadístico de las columnas categóricas.\nstats_cat = df.describe(include=['object'])\n\n# Cantidad de valores únicos en columnas clave.\nunique_values = {\n    'status': df['status'].nunique(),\n    'productline': df['productline'].nunique(),\n    'country': df['country'].nunique(),\n    'territory': df['territory'].nunique(),\n    'dealsize': df['dealsize'].nunique(),\n    'year_id': df['year_id'].nunique()\n}\n\n# Total de ventas y cantidad de pedidos.\ntotal_sales = df['sales'].sum()\ntotal_orders = df['ordernumber'].nunique()\n\n(stats_num, stats_cat, unique_values, total_sales, total_orders)"}`


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 e

In [34]:
respuesta = agent.invoke({
    "input": "¿Cuántas filas y columnas tiene el dataset? Responde solo con el número de filas y columnas."
})

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.shape'}`


(2823, 25)2823 filas y 25 columnas.

> Finished chain.
2823 filas y 25 columnas.


In [35]:
respuesta = agent.invoke({
    "input": "Calcula el total de ventas usando la columna sales. Responde con el monto total."
})

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['sales'].sum()"}`


10032628.85El total de ventas, calculado sumando la columna **sales**, es **10.032.628,85**. Este valor representa la suma de todos los montos de venta registrados en el dataset.

> Finished chain.
El total de ventas, calculado sumando la columna **sales**, es **10.032.628,85**. Este valor representa la suma de todos los montos de venta registrados en el dataset.


In [36]:
respuesta = agent.invoke({
    "input": "Agrupa las ventas por country usando la columna sales y muestra los 5 países con mayores ventas totales."
})

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Agrupar las ventas por país y sumar los montos totales de ventas\nsales_by_country = df.groupby('country')['sales'].sum().reset_index()\n\n# Ordenar los países por ventas totales en orden descendente y seleccionar los 5 primeros\ntop_5_countries = sales_by_country.sort_values(by='sales', ascending=False).head(5)\n\ntop_5_countries"}`


      country       sales
18        USA  3627982.83
14      Spain  1215686.92
6      France  1110916.52
0   Australia   630623.10
17         UK   478880.46**Análisis de los 5 países con mayores ventas totales:**

Se agruparon las ventas totales (`sales`) por país (`country`) y se sumaron los montos para cada uno. Luego, se ordenaron de mayor a menor y se seleccionaron los 5 países con las ventas más altas. Estos son:

1. **USA**: **$3,627,982.83** (Mayor volumen de ventas).
2. **Spain**: **$1,215,686.92**.
3. **France**: **$1,110,916.52**.
4. **Australia**: **$630,623

In [37]:
respuesta = agent.invoke({
    "input": "Basándote en los datos reales del dataset, dame 3 recomendaciones de negocio. Primero calcula ventas por país, ventas por línea de producto y cantidad de pedidos por estado."
})

print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Calcular ventas totales por país\ndf_ventas_pais = df.groupby('country')['sales'].sum().reset_index()\ndf_ventas_pais = df_ventas_pais.sort_values(by='sales', ascending=False)\n\n# Calcular ventas totales por línea de producto\ndf_ventas_producto = df.groupby('productline')['sales'].sum().reset_index()\ndf_ventas_producto = df_ventas_producto.sort_values(by='sales', ascending=False)\n\n# Calcular cantidad de pedidos por estado (status)\ndf_pedidos_estado = df.groupby('status')['ordernumber'].nunique().reset_index()\ndf_pedidos_estado = df_pedidos_estado.sort_values(by='ordernumber', ascending=False)\n\n(df_ventas_pais, df_ventas_producto, df_pedidos_estado)"}`


(        country       sales
18          USA  3627982.83
14        Spain  1215686.92
6        France  1110916.52
0     Australia   630623.10
17           UK   478880.46
9         Italy   374674.31
5       Finland   329581.91
11       Norway 